In [2]:
import pdfplumber

In [3]:
text = ""
with pdfplumber.open("d:/Rag_chatbot/data/AI Training Document.pdf") as pdf:
    for page in pdf.pages:
        text += page.extract_text() + "\n"

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

In [4]:
import re

In [5]:
clean_text=re.sub(r"http\S+|www\.\S+", "",text)

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100,separators=["\n\n"," ","\n"])

In [8]:
chunks = splitter.split_text(clean_text)

### Now we will use sentence transformers all mini lm to get embedding vectors 

In [9]:
from sentence_transformers import SentenceTransformer

In [10]:
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

In [11]:
embeddings=emb_model.encode(chunks)
# we will get emb in vectors form one vector per chunks

### Now we are going to use Faiss for storing those vectors

In [12]:
import faiss

In [17]:
embeddings.shape

(76, 384)

In [18]:
Findex= faiss.IndexFlatL2(embeddings.shape[1]) #each embedding vec is the size of 384 dim

In [19]:
Findex.add(embeddings)

### We are going for Phi-3 mini Lm bcos of its small size and we will do it using ollama

In [20]:
user_query= "give me a short summary of 5 lines "
user_query_vec= emb_model.encode([user_query])

In [21]:
#We are searching here for similar distance between chunks and query
distance,indices =Findex.search(user_query_vec,k=3)

In [22]:
top_chunks=[]
for i in indices[0]:
    top_chunks.append(chunks[i])

In [23]:
context="\n".join(top_chunks)
final_prompt=f"based on following context:{context} answer the following questions:{user_query}"

In [24]:
import ollama

In [25]:
response=ollama.chat(model="phi3:mini",
                     messages=[{"role":"system",
                                "content":"you are a chatbot used for answering users query"},
                                {"role":"user","content":final_prompt}],)
print(response["message"]["content"])

eBay's User Agreement disclaimers cover liability for various damages arising from use or content provided on eBay Services. Warranties, claims based on third-party actions, account suspensions, search result appearance terms, and required business changes due to agreement updates are all addressed in the agreement with exclusions of certain types of losses and limitations related to foreseeability. 

However, these disclaimers might not apply if local laws prohibit their exclusion or limitation; consequently, eBay may still be liable under such jurisdictions' statutes.
